# 08 — TPU Ablation Lab (125M Model)

Controlled ablation experiments on TPU v6e for a **125M non-embed param**
transformer (D1024-F3072-B4, L=8). Selected from Phase 11 benchmarks.

**Three modes:**
1. **Quick Training** (~300 steps, ~7s) — XProf capture, MFU measurement
2. **Sweep** (wandb, ~15-20 min/run) — Bayesian LR search per architecture config
3. **Hero Run** (20 tok/param, ~1.9h) — Full training with eval + optional checkpointing

**Ablation knobs** (set in the Model section):
- `ATTN_IMPL`: `'splash'` or `'einsum'`
- `MLP_TYPE`: `'glu'` (SwiGLU, F=3072) or `'plain'` (ReLU², F=4096)
- `QK_NORM`: `True` or `False`

### Architecture: D=1024, N=3, K=1, H=256, F=3072, L=8, B=4, T=2048, V=32768
| Metric | Value |
|--------|-------|
| Total params | 159.4M |
| Non-embed params | 125.8M |
| Step time (splash) | ~22ms |
| Throughput | ~373k tok/s |
| MFU | ~36.8% |
| Hero run (20 tok/param) | ~2.5B tokens, ~308k steps, ~1.9h |

In [ ]:
!pip install -q "jax[tpu]" optax huggingface_hub tiktoken pyarrow requests wandb tensorboard tensorboard-plugin-profile

## Utilities

In [ ]:
# === Imports, TPU constants, dot_dict ===
import functools as ft
import time
import os
import math
import pickle
import queue
import threading
from dataclasses import dataclass

import jax
import jax.numpy as jnp
import numpy as np
import optax
import tiktoken

# TPU v6e-1 constants
PEAK_TFLOPS = 918          # bf16 peak compute per chip
HBM_GB = 32
MXU_DIM = 256              # 256×256 systolic array

print(f"JAX version : {jax.__version__}")
print(f"Devices     : {jax.devices()}")
print(f"Peak TFLOPS : {PEAK_TFLOPS} (bf16, from v6e docs)")


@jax.tree_util.register_pytree_with_keys_class
class dot_dict(dict):
    __setattr__ = dict.__setitem__
    __getattr__ = dict.__getitem__

    def tree_flatten_with_keys(self):
        keys = tuple(sorted(self))
        return tuple((jax.tree_util.DictKey(k), self[k]) for k in keys), keys

    @classmethod
    def tree_unflatten(cls, keys, values):
        return cls(zip(keys, values))

In [ ]:
# === RMSNorm, RoPE ===

def rms_norm(x):
    """RMSNorm with no learnable parameters."""
    return x * jax.lax.rsqrt(jnp.mean(x * x, axis=-1, keepdims=True) + 1e-6)


def precompute_rope(seq_len, head_dim, base=10000):
    """Precompute rotary embedding cos/sin tables."""
    channel_range = jnp.arange(0, head_dim, 2, dtype=jnp.float32)
    inv_freq = 1.0 / (base ** (channel_range / head_dim))
    t = jnp.arange(seq_len, dtype=jnp.float32)
    freqs = jnp.outer(t, inv_freq)
    cos = jnp.cos(freqs).astype(jnp.bfloat16)
    sin = jnp.sin(freqs).astype(jnp.bfloat16)
    return cos, sin


def apply_rope(x, cos, sin):
    """Apply rotary embeddings. x: (B, H, T, D), cos/sin: (1, 1, T, D/2)"""
    d = x.shape[-1] // 2
    x1, x2 = x[..., :d], x[..., d:]
    y1 = x1 * cos + x2 * sin
    y2 = x1 * (-sin) + x2 * cos
    return jnp.concatenate([y1, y2], axis=-1)

In [ ]:
# === Attention backends (einsum + splash) ===

def _expand_kv(k, v, n_head, n_kv_head):
    """Repeat KV heads to match Q head count for non-splash backends."""
    if n_kv_head == n_head:
        return k, v
    ratio = n_head // n_kv_head
    return jnp.repeat(k, ratio, axis=1), jnp.repeat(v, ratio, axis=1)

In [ ]:
# === FLOP counting helpers ===
# Dimension notation: B=batch, T=seq_len, D=d_model, N=n_heads,
#   K=n_kv_heads, H=head_dim, F=d_ff, L=n_layers, V=vocab_size

def matmul_flops(M, N, K, batch=1):
    """FLOPs for [M,K] @ [K,N].  2*M*N*K per batch element."""
    return 2 * batch * M * N * K


def attention_flops(B, N, T, H):
    """FLOPs for QK^T + AV (full T×T, not causal-halved)."""
    return 2 * (2 * B * N * T * T * H)


def layer_flops(B, T, D, N, K, H, F, mlp_type='glu'):
    """MXU-relevant FLOPs for one transformer layer.

    Counts only matmul FLOPs (projections + attention core + MLP).
    mlp_type='glu': 3 MLP matmuls (gate + up + down).
    mlp_type='plain': 2 MLP matmuls (up + down).
    """
    tok = B * T
    q    = 2 * tok * D * N * H          # Q projection
    k    = 2 * tok * D * K * H          # K projection
    v    = 2 * tok * D * K * H          # V projection
    att  = attention_flops(B, N, T, H)  # core attention
    proj = 2 * tok * N * H * D          # output projection
    if mlp_type == 'glu':
        mlp = 3 * (2 * tok * D * F)     # gate + up + down
    else:
        mlp = 2 * (2 * tok * D * F)     # up + down
    return q + k + v + att + proj + mlp

In [ ]:
# === Data: HF login, tokenizer, data download, tokenize_shards, PrefetchDataLoader ===
import requests
from multiprocessing import Pool
import pyarrow.parquet as pq
from huggingface_hub import login, hf_hub_download

HF_REPO_ID = 'vorushin/tpuchat'
DATA_DIR = '/content/base_data'
TOKENIZER_DIR = '/content/tokenizer'
MAX_CHARS_PER_DOC = 10_000
NUM_TRAIN_SHARDS = 50
NUM_VAL_SHARDS = 2

# --- HF login + tokenizer ---
login()

os.makedirs(TOKENIZER_DIR, exist_ok=True)
hf_hub_download(repo_id=HF_REPO_ID, filename='tokenizer/tokenizer.pkl',
                local_dir=TOKENIZER_DIR)
print(f'Downloaded tokenizer to {TOKENIZER_DIR}')

with open(os.path.join(TOKENIZER_DIR, 'tokenizer', 'tokenizer.pkl'), 'rb') as f:
    enc = pickle.load(f)
print(f'Loaded tokenizer: vocab_size={enc.n_vocab}')

# --- Download data shards ---
BASE_URL = 'https://huggingface.co/datasets/karpathy/fineweb-edu-100b-shuffle/resolve/main'
os.makedirs(DATA_DIR, exist_ok=True)

def download_shard(index):
    filename = f'shard_{index:05d}.parquet'
    filepath = os.path.join(DATA_DIR, filename)
    if os.path.exists(filepath):
        return True
    url = f'{BASE_URL}/{filename}'
    print(f'Downloading {filename}...')
    for attempt in range(1, 4):
        try:
            resp = requests.get(url, stream=True, timeout=60)
            resp.raise_for_status()
            tmp = filepath + '.tmp'
            with open(tmp, 'wb') as f:
                for chunk in resp.iter_content(1024 * 1024):
                    if chunk:
                        f.write(chunk)
            os.rename(tmp, filepath)
            return True
        except Exception as e:
            print(f'Attempt {attempt}/3 failed for {filename}: {e}')
            for p in [filepath + '.tmp', filepath]:
                if os.path.exists(p):
                    os.remove(p)
            if attempt < 3:
                time.sleep(2 ** attempt)
    return False

total_shards = NUM_TRAIN_SHARDS + NUM_VAL_SHARDS
t0 = time.time()
with Pool(8) as pool:
    results = pool.map(download_shard, range(total_shards))
print(f'\nDownloaded {sum(results)}/{total_shards} shards in {time.time()-t0:.1f}s')


# --- Tokenize shards ---
def tokenize_shards(shard_indices, batch_size, seq_len):
    """Yield (x, y) batches by tokenizing parquet shards on the fly."""
    bos_id = enc.encode_single_token('<|bos|>')
    buf = []

    while True:  # loop over epochs
        for shard_idx in shard_indices:
            filepath = os.path.join(DATA_DIR, f'shard_{shard_idx:05d}.parquet')
            pf = pq.ParquetFile(filepath)
            for rg_idx in range(pf.num_row_groups):
                rg = pf.read_row_group(rg_idx)
                texts = rg.column('text').to_pylist()
                for doc in texts:
                    if len(doc) > MAX_CHARS_PER_DOC:
                        doc = doc[:MAX_CHARS_PER_DOC]
                    tokens = [bos_id] + enc.encode_ordinary(doc)
                    buf.extend(tokens)

                    tokens_per_batch = batch_size * (seq_len + 1)
                    while len(buf) >= tokens_per_batch:
                        batch_tokens = np.array(buf[:tokens_per_batch], dtype=np.int32)
                        batch_tokens = batch_tokens.reshape(batch_size, seq_len + 1)
                        x = batch_tokens[:, :-1]
                        y = batch_tokens[:, 1:]
                        buf = buf[tokens_per_batch:]
                        yield x, y


train_shard_indices = list(range(NUM_TRAIN_SHARDS))
val_shard_indices = list(range(NUM_TRAIN_SHARDS, NUM_TRAIN_SHARDS + NUM_VAL_SHARDS))
print(f'Train shards: {len(train_shard_indices)}, Val shards: {len(val_shard_indices)}')


# --- PrefetchDataLoader ---
@dataclass
class PrefetchDataLoader:
    """Wraps an iterator and prefetches items in a background thread."""
    iterator: any
    capacity: int = 2

    def __post_init__(self):
        self.queue = queue.Queue(maxsize=self.capacity)
        self.stop_event = threading.Event()
        self.thread = threading.Thread(target=self._worker, daemon=True)
        self.thread.start()

    def _worker(self):
        try:
            for item in self.iterator:
                if self.stop_event.is_set():
                    break
                x, y = item
                item = (jax.device_put(jnp.array(x)), jax.device_put(jnp.array(y)))
                self.queue.put(item)
        except Exception as e:
            print(f"Prefetch worker error: {e}")
            self.stop_event.set()
        finally:
            self.stop_event.set()

    def __iter__(self):
        return self

    def __next__(self):
        if self.stop_event.is_set() and self.queue.empty():
            raise StopIteration
        return self.queue.get()

    def stop(self):
        self.stop_event.set()

In [ ]:
# === Optimizer: AdamW with warmup + linear warmdown ===

def init_adam_state(param):
    """Initialize Adam optimizer state for a single parameter."""
    return dot_dict(
        mu=jnp.zeros_like(param),
        nu=jnp.zeros_like(param),
        count=jnp.array(0, dtype=jnp.int32),
    )


def adamw_step(config, lr_mult, param, grad, state):
    """AdamW update. Returns (new_param, new_state)."""
    new_count = state.count + 1
    new_mu = config.beta1 * state.mu + (1 - config.beta1) * grad
    new_nu = config.beta2 * state.nu + (1 - config.beta2) * grad ** 2

    mu_hat = new_mu / (1 - config.beta1 ** new_count)
    nu_hat = new_nu / (1 - config.beta2 ** new_count)

    lr = config.learning_rate * lr_mult
    update = mu_hat / (jnp.sqrt(nu_hat) + config.eps)

    # Weight decay for 2D+ params
    wd = jnp.where(param.ndim >= 2, config.weight_decay, 0.0)
    new_param = param - lr * (update + wd * param)

    new_state = dot_dict(mu=new_mu, nu=new_nu, count=new_count)
    return new_param, new_state


def get_lr_multiplier(step, num_iterations, config):
    """Linear warmup, constant, linear warmdown schedule."""
    warmup_iters = int(config.warmup_ratio * num_iterations)
    warmdown_iters = int(config.warmdown_ratio * num_iterations)

    if step < warmup_iters:
        return (step + 1) / max(warmup_iters, 1)
    elif step <= num_iterations - warmdown_iters:
        return 1.0
    else:
        progress = (num_iterations - step) / max(warmdown_iters, 1)
        return progress * 1.0 + (1 - progress) * config.final_lr_frac

In [ ]:
# === split_trainable, merge_params, count_params, count_non_embed_params ===

def split_trainable(params):
    """Split params into trainable and static (non-differentiable)."""
    trainable = dot_dict()
    static = dot_dict()
    for k, v in params.items():
        if k in ('rope_cos', 'rope_sin'):
            static[k] = v
        else:
            trainable[k] = v
    return trainable, static


def merge_params(trainable, static):
    """Merge trainable and static params back together."""
    merged = dot_dict()
    merged.update(trainable)
    merged.update(static)
    return merged


def count_params(params):
    """Count total trainable parameters (excludes rope_cos/rope_sin)."""
    trainable, _ = split_trainable(params)
    return sum(p.size for p in jax.tree.leaves(trainable) if isinstance(p, jax.Array))


def count_non_embed_params(params):
    """Non-embedding params (unembed + layers). Excludes wte (lookup table)."""
    return count_params(params) - params.wte.size

## Model

In [ ]:
# === Ablation knobs + fixed constants + Config ===

# ── Ablation knobs ─────────────────────────────────────────────
ATTN_IMPL = 'splash'    # 'splash' | 'einsum'
MLP_TYPE  = 'glu'       # 'glu' (SwiGLU, F=3D=3072) | 'plain' (ReLU², F=4D=4096)
QK_NORM   = True         # QK-norm on queries and keys

# ── Fixed architecture (D1024, 125M non-embed) ────────────────
D, L, T, V = 1024, 8, 2048, 32768
N_HEAD, N_KV_HEAD, HEAD_DIM = 3, 1, 256
F_GLU, F_PLAIN = 3072, 4096     # MLP width depends on MLP_TYPE
SOFTCAP = 15.0
SPLASH_BS, LM_CHUNKS = 1024, 8
BATCH_SIZE = 4

# ── Training defaults ─────────────────────────────────────────
LR = 3e-4
BETA1, BETA2, EPS, WD = 0.9, 0.95, 1e-8, 0.1
WARMUP_RATIO, WARMDOWN_RATIO = 0.02, 0.5
NUM_SHARDS = 50


@jax.tree_util.register_static
@dataclass(kw_only=True, frozen=True)
class Config:
    # Architecture
    n_embd: int
    n_layer: int
    seq_len: int
    vocab_size: int
    n_head: int
    n_kv_head: int
    head_dim: int
    mlp_dim: int
    mlp_type: str
    attn_impl: str
    qk_norm: bool
    softcap: float
    splash_block_size: int
    num_lm_head_chunks: int
    batch_size: int

    # Training
    learning_rate: float
    beta1: float
    beta2: float
    eps: float
    weight_decay: float
    warmup_ratio: float
    warmdown_ratio: float
    final_lr_frac: float = 0.0

    # Eval / Data
    eval_steps: int = 10
    param_seed: int = 42

    @property
    def padded_vocab(self):
        return ((self.vocab_size + 63) // 64) * 64


def make_config(lr=LR, **overrides):
    """Build Config from module-level constants + optional overrides."""
    defaults = dict(
        n_embd=D, n_layer=L, seq_len=T, vocab_size=V,
        n_head=N_HEAD, n_kv_head=N_KV_HEAD, head_dim=HEAD_DIM,
        mlp_dim=F_GLU if MLP_TYPE == 'glu' else F_PLAIN,
        mlp_type=MLP_TYPE, attn_impl=ATTN_IMPL, qk_norm=QK_NORM,
        softcap=SOFTCAP, splash_block_size=SPLASH_BS,
        num_lm_head_chunks=LM_CHUNKS, batch_size=BATCH_SIZE,
        learning_rate=lr, beta1=BETA1, beta2=BETA2, eps=EPS,
        weight_decay=WD, warmup_ratio=WARMUP_RATIO,
        warmdown_ratio=WARMDOWN_RATIO,
    )
    defaults.update(overrides)
    return Config(**defaults)


config = make_config()
print(f'Config: D={config.n_embd}, L={config.n_layer}, T={config.seq_len}, '
      f'V={config.vocab_size}, N={config.n_head}, K={config.n_kv_head}, '
      f'H={config.head_dim}, F={config.mlp_dim}')
print(f'Ablations: attn_impl={config.attn_impl}, mlp_type={config.mlp_type}, '
      f'qk_norm={config.qk_norm}')
print(f'Training: lr={config.learning_rate:.1e}, B={config.batch_size}')

In [ ]:
# === init_layer_params (branches on mlp_type) ===

def init_layer_params(config, seed=42):
    """Initialize params for one transformer layer."""
    key = jax.random.key(seed)
    keys = jax.random.split(key, 7)
    s = (3.0 ** 0.5) * (config.n_embd ** -0.5)
    layer = dot_dict()

    # Attention projections
    layer.c_q = jax.random.uniform(keys[0], (config.n_embd, config.n_head, config.head_dim),
                                    dtype=jnp.bfloat16, minval=-s, maxval=s)
    layer.c_k = jax.random.uniform(keys[1], (config.n_embd, config.n_kv_head, config.head_dim),
                                    dtype=jnp.bfloat16, minval=-s, maxval=s)
    layer.c_v = jax.random.uniform(keys[2], (config.n_embd, config.n_kv_head, config.head_dim),
                                    dtype=jnp.bfloat16, minval=-s, maxval=s)
    layer.c_proj = jnp.zeros((config.n_head, config.head_dim, config.n_embd), dtype=jnp.bfloat16)

    # MLP — shape depends on mlp_type
    if config.mlp_type == 'glu':
        # SwiGLU: gate (D,F) + up (D,F) + down (F,D)
        layer.w_gate = jax.random.uniform(keys[3], (config.n_embd, config.mlp_dim),
                                           dtype=jnp.bfloat16, minval=-s, maxval=s)
        layer.w_up = jax.random.uniform(keys[4], (config.n_embd, config.mlp_dim),
                                         dtype=jnp.bfloat16, minval=-s, maxval=s)
        layer.w_down = jnp.zeros((config.mlp_dim, config.n_embd), dtype=jnp.bfloat16)
    else:
        # Plain (ReLU²): up (D,F) + down (F,D)
        layer.w_up = jax.random.uniform(keys[3], (config.n_embd, config.mlp_dim),
                                         dtype=jnp.bfloat16, minval=-s, maxval=s)
        layer.w_down = jnp.zeros((config.mlp_dim, config.n_embd), dtype=jnp.bfloat16)
    return layer

In [ ]:
# === single_layer_forward (uses attn_impl, qk_norm, mlp_type from config) ===

def single_layer_forward(config, layer, x, cos, sin, layer_idx=0):
    """Forward pass for one transformer layer."""
    h = rms_norm(x)

    with jax.named_scope(f'layer_{layer_idx}/attention'):
        # --- Attention ---
        q = jnp.einsum('btd,dhk->bhtk', h, layer.c_q)
        k = jnp.einsum('btd,dhk->bhtk', h, layer.c_k)
        v = jnp.einsum('btd,dhk->bhtk', h, layer.c_v)

        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        if config.qk_norm:
            q = rms_norm(q)
            k = rms_norm(k)

        seq_len = x.shape[1]

        if config.attn_impl == 'splash':
            from jax.experimental.pallas.ops.tpu.splash_attention import (
                splash_attention_mask, splash_attention_kernel)

            smask = splash_attention_mask.CausalMask(shape=(seq_len, seq_len))
            mh_mask = splash_attention_mask.MultiHeadMask(
                masks=[smask] * config.n_head)
            bs = min(config.splash_block_size, seq_len)
            block_sizes = splash_attention_kernel.BlockSizes(
                block_q=bs, block_kv=bs,
                block_q_dkv=bs, block_kv_dkv=bs,
                block_q_dq=bs, block_kv_dq=bs)
            kernel = splash_attention_kernel.make_splash_mha(
                mask=mh_mask, head_shards=1, q_seq_shards=1,
                block_sizes=block_sizes)
            attn_out = jax.vmap(kernel)(q, k, v)

        elif config.attn_impl == 'einsum':
            k_exp, v_exp = _expand_kv(k, v, config.n_head, config.n_kv_head)
            scale = config.head_dim ** -0.5
            scores = jnp.einsum('bhtd,bhsd->bhts', q, k_exp) * scale
            rows = jnp.arange(seq_len)[:, None]
            cols = jnp.arange(seq_len)[None, :]
            mask = cols <= rows
            scores = jnp.where(mask[None, None, :, :], scores,
                               jnp.finfo(scores.dtype).min)
            attn_weights = jax.nn.softmax(scores, axis=-1)
            attn_out = jnp.einsum('bhts,bhsd->bhtd', attn_weights, v_exp)

        attn_out = jnp.einsum('bhtd,hde->bte', attn_out, layer.c_proj)

    x = x + attn_out

    with jax.named_scope(f'layer_{layer_idx}/mlp'):
        # --- MLP ---
        h2 = rms_norm(x)
        if config.mlp_type == 'glu':
            gate = jax.nn.silu(jnp.einsum('btd,dh->bth', h2, layer.w_gate))
            up = jnp.einsum('btd,dh->bth', h2, layer.w_up)
            mlp_out = jnp.einsum('bth,hd->btd', gate * up, layer.w_down)
        else:  # plain (ReLU²)
            mlp_out = jnp.einsum('btd,dh->bth', h2, layer.w_up)
            mlp_out = jax.nn.relu(mlp_out) ** 2
            mlp_out = jnp.einsum('bth,hd->btd', mlp_out, layer.w_down)

    x = x + mlp_out
    return x

In [ ]:
# === init_full_model, model_forward, model_forward_remat ===

def init_all_layers(config, n_layers, seed=42):
    layers = dot_dict()
    for i in range(n_layers):
        layers[i] = init_layer_params(config, seed=seed + i * 7)
    return layers


def init_full_model(config, seed=42):
    """Initialize all model params (embed + layers + lm_head + rope)."""
    key = jax.random.key(seed)
    params = dot_dict()
    key, k1, k2 = jax.random.split(key, 3)
    params.wte = jax.random.normal(k1, (config.padded_vocab, config.n_embd),
                                    dtype=jnp.bfloat16)
    params.lm_head = jax.random.normal(k2, (config.n_embd, config.padded_vocab),
                                        dtype=jnp.bfloat16) * 0.001
    params.rope_cos, params.rope_sin = precompute_rope(config.seq_len, config.head_dim)
    params.layers = init_all_layers(config, config.n_layer, seed=seed + 100)
    return params


def model_forward(config, params, tokens):
    """Full forward: embed -> layers -> final_norm. Returns hidden (B,T,D)."""
    B, T = tokens.shape
    cos = params.rope_cos[:T][None, None, :, :]
    sin = params.rope_sin[:T][None, None, :, :]
    with jax.named_scope('embedding'):
        x = rms_norm(params.wte[tokens])
    for i in range(config.n_layer):
        x = single_layer_forward(config, params.layers[i], x, cos, sin, layer_idx=i)
    return rms_norm(x)


def model_forward_remat(config, params, tokens):
    """Same as model_forward but with jax.checkpoint on each layer."""
    B, T = tokens.shape
    cos = params.rope_cos[:T][None, None, :, :]
    sin = params.rope_sin[:T][None, None, :, :]
    with jax.named_scope('embedding'):
        x = rms_norm(params.wte[tokens])
    for i in range(config.n_layer):
        layer_fn = ft.partial(single_layer_forward, config, layer_idx=i)
        x = jax.checkpoint(layer_fn)(params.layers[i], x, cos, sin)
    return rms_norm(x)

In [ ]:
# === chunked_lm_head_loss ===

def _logits_from_chunk(h_chunk, lm_head, config):
    logits = jnp.einsum('td,dv->tv', h_chunk, lm_head)
    logits = logits[:, :config.vocab_size]
    logits = logits.astype(jnp.float32)
    return config.softcap * jnp.tanh(logits / config.softcap)


@ft.partial(jax.custom_vjp, nondiff_argnums=(3,))
def chunked_lm_head_loss(hidden, lm_head, labels, config):
    B, T, D = hidden.shape
    N = config.num_lm_head_chunks
    S = B * T // N
    hidden_chunks = hidden.reshape(N, S, D)
    labels_chunks = labels.reshape(N, S)

    def fwd_body(_, data):
        h_chunk, l_chunk = data
        return None, jnp.sum(
            optax.softmax_cross_entropy_with_integer_labels(
                _logits_from_chunk(h_chunk, lm_head, config), l_chunk))

    _, chunk_losses = jax.lax.scan(fwd_body, None, (hidden_chunks, labels_chunks))
    return jnp.sum(chunk_losses) / (B * T)


def _chunked_loss_fwd(hidden, lm_head, labels, config):
    loss = chunked_lm_head_loss(hidden, lm_head, labels, config)
    return loss, (hidden, lm_head, labels)


def _chunked_loss_bwd(config, residuals, g):
    hidden, lm_head, labels = residuals
    B, T, D = hidden.shape
    N = config.num_lm_head_chunks
    S = B * T // N
    hidden_chunks = hidden.reshape(N, S, D)
    labels_chunks = labels.reshape(N, S)

    def bwd_body(d_lm_head_acc, data):
        h_chunk, l_chunk = data

        def chunk_loss(h, w):
            return jnp.sum(
                optax.softmax_cross_entropy_with_integer_labels(
                    _logits_from_chunk(h, w, config), l_chunk))

        _, vjp_fn = jax.vjp(chunk_loss, h_chunk, lm_head)
        d_h, d_w = vjp_fn(g / (B * T))
        return d_lm_head_acc + d_w, d_h

    d_lm_head_init = jnp.zeros_like(lm_head)
    d_lm_head, d_hidden_chunks = jax.lax.scan(
        bwd_body, d_lm_head_init, (hidden_chunks, labels_chunks))
    return d_hidden_chunks.reshape(B, T, D), d_lm_head, jnp.zeros_like(labels)


chunked_lm_head_loss.defvjp(_chunked_loss_fwd, _chunked_loss_bwd)

In [ ]:
# === train_step, eval_step, predict_step, generate ===

@jax.jit
def train_step(config, params, opt_state, x, y, lr_mult):
    """Single training step: forward, backward, optimizer update."""
    trainable, static = split_trainable(params)

    def loss_fn(trainable_params):
        full_params = merge_params(trainable_params, static)
        hidden = model_forward_remat(config, full_params, x)
        return chunked_lm_head_loss(hidden, full_params.lm_head, y, config)

    with jax.named_scope('forward_backward'):
        loss, grads = jax.value_and_grad(loss_fn)(trainable)

    with jax.named_scope('optimizer'):
        is_opt_leaf = lambda x: isinstance(x, dot_dict) and 'mu' in x
        t_leaves, t_treedef = jax.tree.flatten(trainable)
        g_leaves, _ = jax.tree.flatten(grads)
        o_leaves, o_treedef = jax.tree.flatten(opt_state, is_leaf=is_opt_leaf)

        new_t_leaves, new_o_leaves = [], []
        for p, g, s in zip(t_leaves, g_leaves, o_leaves):
            new_p, new_s = adamw_step(config, lr_mult, p, g, s)
            new_t_leaves.append(new_p)
            new_o_leaves.append(new_s)

        new_trainable = t_treedef.unflatten(new_t_leaves)
        new_opt_state = o_treedef.unflatten(new_o_leaves)
        new_params = merge_params(new_trainable, static)

    return loss, new_params, new_opt_state


@jax.jit
def eval_step(config, params, x, y):
    """JIT-compiled eval: returns loss for a single batch."""
    hidden = model_forward(config, params, x)
    return chunked_lm_head_loss(hidden, params.lm_head, y, config)


@jax.jit
def predict_step(config, params, x):
    """JIT-compiled single step inference: returns logits."""
    hidden = model_forward(config, params, x)
    with jax.named_scope('lm_head'):
        logits = jnp.einsum('btd,dv->btv', hidden, params.lm_head)
        logits = logits[:, :, :config.vocab_size]
        logits = logits.astype(jnp.float32)
        logits = config.softcap * jnp.tanh(logits / config.softcap)
    return logits


def generate(config, params, prompt, max_new_tokens=64, temperature=0.8):
    """Generate text from a prompt using temperature sampling."""
    bos_id = enc.encode_single_token('<|bos|>')
    ids = [bos_id] + enc.encode_ordinary(prompt)
    key = jax.random.key(42)

    for _ in range(max_new_tokens):
        context = ids[-config.seq_len:]
        pad_len = config.seq_len - len(context)
        x = jnp.array([context + [0] * pad_len], dtype=jnp.int32)
        logits = predict_step(config, params, x)
        logits.block_until_ready()
        next_logits = logits[0, len(context) - 1, :]

        if temperature == 0:
            next_id = int(jnp.argmax(next_logits))
        else:
            key, subkey = jax.random.split(key)
            next_logits = next_logits / temperature
            next_id = int(jax.random.categorical(subkey, next_logits))
        ids.append(next_id)

    return enc.decode(ids)

## Quick Training (XProf)

In [ ]:
# === Quick Training: ~300 steps, XProf on 15-20, MFU measurement ===

NUM_QUICK_STEPS = 300
EVAL_EVERY = 100
XPROF_START, XPROF_END = 15, 20
LOG_DIR = '/content/log_dir'

# Init model + optimizer
params = init_full_model(config, seed=config.param_seed)
total_p = count_params(params)
non_embed_p = count_non_embed_params(params)
print(f'Params: {total_p/1e6:.1f}M total, {non_embed_p/1e6:.1f}M non-embed')
print(f'Batch: {config.batch_size} x {config.seq_len} = '
      f'{config.batch_size * config.seq_len:,} tokens/step')

trainable_params, static_params = split_trainable(params)
opt_state = jax.tree.map(init_adam_state, trainable_params)

# Data
raw_train = tokenize_shards(train_shard_indices, config.batch_size, config.seq_len)
train_loader = PrefetchDataLoader(raw_train, capacity=4)
val_loader_fn = lambda: tokenize_shards(val_shard_indices, config.batch_size, config.seq_len)

# FLOP counting
fwd_flops = (config.n_layer * layer_flops(
    config.batch_size, config.seq_len, config.n_embd,
    config.n_head, config.n_kv_head, config.head_dim,
    config.mlp_dim, config.mlp_type)
    + matmul_flops(config.batch_size * config.seq_len,
                   config.padded_vocab, config.n_embd))
step_flops = 3 * fwd_flops  # fwd + 2x bwd

smooth_loss = 0.0
step_times = []

print(f'\n=== Quick Training: {NUM_QUICK_STEPS} steps ===\n')

for step in range(NUM_QUICK_STEPS + 1):
    last_step = (step == NUM_QUICK_STEPS)

    # --- Eval ---
    if step % EVAL_EVERY == 0 or last_step:
        val_loader = val_loader_fn()
        val_losses = []
        for ei in range(config.eval_steps):
            vx, vy = next(val_loader)
            vx, vy = jnp.array(vx), jnp.array(vy)
            vl = eval_step(config, params, vx, vy)
            val_losses.append(float(vl))
        avg_val_loss = sum(val_losses) / len(val_losses)
        print(f'Step {step:05d} | Val loss: {avg_val_loss:.4f}')

    if last_step:
        break

    # --- XProf ---
    if step == XPROF_START:
        jax.profiler.start_trace(LOG_DIR)
        print("XProf started...")
    if step == XPROF_END:
        jax.profiler.stop_trace()
        print(f"XProf stopped. Trace saved to '{LOG_DIR}'.")

    # --- Train step ---
    lr_mult = jnp.array(get_lr_multiplier(step, NUM_QUICK_STEPS, config),
                         dtype=jnp.float32)
    t0 = time.time()
    x_batch, y_batch = next(train_loader)
    loss, params, opt_state = train_step(config, params, opt_state,
                                          x_batch, y_batch, lr_mult)
    loss.block_until_ready()
    dt = time.time() - t0

    if step > XPROF_END:
        step_times.append(dt)

    loss_val = float(loss)
    ema_beta = 0.9
    smooth_loss = ema_beta * smooth_loss + (1 - ema_beta) * loss_val
    debiased_loss = smooth_loss / (1 - ema_beta ** (step + 1))

    if step % 50 == 0:
        tok_per_sec = int(config.batch_size * config.seq_len / dt) if dt > 0 else 0
        print(f'step {step:05d}/{NUM_QUICK_STEPS} | loss: {debiased_loss:.4f} '
              f'| dt: {dt*1000:.0f}ms | tok/s: {tok_per_sec:,}')

train_loader.stop()

# --- MFU report ---
if step_times:
    avg_dt = sum(step_times) / len(step_times)
    mfu_pct = step_flops / (PEAK_TFLOPS * 1e12 * avg_dt) * 100
    tok_per_s = int(config.batch_size * config.seq_len / avg_dt)
    ideal_tok_s = PEAK_TFLOPS * 1e12 / (step_flops / (config.batch_size * config.seq_len))
    print(f'\nMFU: {mfu_pct:.1f}% | tok/s: {tok_per_s:,} | '
          f'ideal tok/s (100% MFU): {int(ideal_tok_s):,}')

# --- Sample text ---
print('\n--- Samples ---')
for prompt in ['The capital of France is', 'Machine learning is']:
    text = generate(config, params, prompt, max_new_tokens=64)
    print(f'Prompt: {prompt}\nOutput: {text}\n')

## Sweep (wandb)

In [ ]:
# === wandb LR sweep ===
# Workflow: set ATTN_IMPL, MLP_TYPE, QK_NORM in the Model section above,
# re-run Model cells, then run this cell to sweep LR for that config.
import wandb

wandb.login()

sweep_config = {
    "name": f"ablation-{MLP_TYPE}-{ATTN_IMPL}-qknorm{QK_NORM}",
    "method": "bayes",
    "metric": {"goal": "minimize", "name": "val_loss"},
    "parameters": {
        "learning_rate": {"distribution": "log_uniform_values",
                          "min": 5e-5, "max": 1e-3},
    },
}

SWEEP_STEPS = 40_000       # ~15 min at 22ms/step
SWEEP_EVAL_EVERY = 2000


def sweep_train_fn():
    """Single training run within a wandb sweep."""
    run = wandb.init()
    lr = wandb.config.learning_rate

    cfg = make_config(lr=lr)
    print(f'Sweep run: lr={lr:.2e}, attn={cfg.attn_impl}, '
          f'mlp={cfg.mlp_type}, qk_norm={cfg.qk_norm}')

    wandb.define_metric("train/loss", step_metric="step")
    wandb.define_metric("train/tok_per_sec", step_metric="step")
    wandb.define_metric("val/loss", step_metric="step")
    wandb.define_metric("val_loss", step_metric="step")

    # Init
    params = init_full_model(cfg, seed=cfg.param_seed)
    total_p = count_params(params)
    non_embed_p = count_non_embed_params(params)
    print(f'Params: {total_p/1e6:.1f}M total, {non_embed_p/1e6:.1f}M non-embed')

    trainable, static = split_trainable(params)
    opt_state = jax.tree.map(init_adam_state, trainable)

    raw_train = tokenize_shards(train_shard_indices, cfg.batch_size, cfg.seq_len)
    train_loader = PrefetchDataLoader(raw_train, capacity=4)
    val_loader_fn = lambda: tokenize_shards(val_shard_indices, cfg.batch_size, cfg.seq_len)

    total_batch_size = cfg.batch_size * cfg.seq_len
    smooth_loss = 0.0
    best_val_loss = float('inf')

    print(f'\n=== Sweep run: {SWEEP_STEPS} steps ===\n')

    try:
        for step in range(SWEEP_STEPS + 1):
            last_step = (step == SWEEP_STEPS)

            # --- Eval ---
            if step % SWEEP_EVAL_EVERY == 0 or last_step:
                val_loader = val_loader_fn()
                val_losses = []
                for ei in range(cfg.eval_steps):
                    vx, vy = next(val_loader)
                    vx, vy = jnp.array(vx), jnp.array(vy)
                    vl = eval_step(cfg, params, vx, vy)
                    val_losses.append(float(vl))
                avg_val_loss = sum(val_losses) / len(val_losses)
                if avg_val_loss < best_val_loss:
                    best_val_loss = avg_val_loss

                wandb.log({
                    "step": step,
                    "val/loss": avg_val_loss,
                    "val_loss": avg_val_loss,
                })
                print(f'Step {step:05d} | Val loss: {avg_val_loss:.4f} '
                      f'(best: {best_val_loss:.4f})')

            if last_step:
                break

            # --- Train ---
            lr_mult = jnp.array(get_lr_multiplier(step, SWEEP_STEPS, cfg),
                                 dtype=jnp.float32)
            t0 = time.time()
            x_batch, y_batch = next(train_loader)
            loss, params, opt_state = train_step(cfg, params, opt_state,
                                                  x_batch, y_batch, lr_mult)
            loss.block_until_ready()
            dt = time.time() - t0

            loss_val = float(loss)
            ema_beta = 0.9
            smooth_loss = ema_beta * smooth_loss + (1 - ema_beta) * loss_val
            debiased_loss = smooth_loss / (1 - ema_beta ** (step + 1))

            if step % 500 == 0:
                tok_per_sec = int(total_batch_size / dt) if dt > 0 else 0
                wandb.log({
                    "step": step,
                    "train/loss": debiased_loss,
                    "train/tok_per_sec": tok_per_sec,
                })
                print(f'step {step:05d}/{SWEEP_STEPS} | loss: {debiased_loss:.4f} '
                      f'| tok/s: {tok_per_sec:,}')

    finally:
        train_loader.stop()

    wandb.finish()
    print(f'Run complete. Best val loss: {best_val_loss:.4f}')


sweep_id = wandb.sweep(sweep_config, project="tpuchat-ablations")
print(f"Sweep ID: {sweep_id}")
wandb.agent(sweep_id, function=sweep_train_fn, count=5)

## Hero Run (20 tok/param)

In [ ]:
# === Hero run: 20 tok/param, ~1.9 hours ===
import wandb

SAVE_CHECKPOINTS = False   # save params to disk every 50k steps
CHECKPOINT_DIR = '/content/checkpoints'

# Compute steps from tok/param ratio
hero_config = make_config()
hero_params = init_full_model(hero_config, seed=hero_config.param_seed)
hero_non_embed = count_non_embed_params(hero_params)
hero_total_p = count_params(hero_params)

target_tokens = int(20 * hero_non_embed)
total_batch_size = hero_config.batch_size * hero_config.seq_len
HERO_STEPS = target_tokens // total_batch_size
HERO_EVAL_EVERY = 1000

print(f'Params: {hero_total_p/1e6:.1f}M total, {hero_non_embed/1e6:.1f}M non-embed')
print(f'Target tokens: {target_tokens:,} (20 tok/param)')
print(f'Steps: {HERO_STEPS:,} ({total_batch_size:,} tok/step)')
print(f'Estimated time: {HERO_STEPS * 0.022 / 3600:.1f} hours (at 22ms/step)')

# Init optimizer
trainable, static = split_trainable(hero_params)
opt_state = jax.tree.map(init_adam_state, trainable)

# Data
raw_train = tokenize_shards(train_shard_indices, hero_config.batch_size, hero_config.seq_len)
train_loader = PrefetchDataLoader(raw_train, capacity=4)
val_loader_fn = lambda: tokenize_shards(val_shard_indices, hero_config.batch_size, hero_config.seq_len)

# FLOP counting for MFU
fwd_flops = (hero_config.n_layer * layer_flops(
    hero_config.batch_size, hero_config.seq_len, hero_config.n_embd,
    hero_config.n_head, hero_config.n_kv_head, hero_config.head_dim,
    hero_config.mlp_dim, hero_config.mlp_type)
    + matmul_flops(hero_config.batch_size * hero_config.seq_len,
                   hero_config.padded_vocab, hero_config.n_embd))
step_flops = 3 * fwd_flops

# wandb
wandb.login()
wandb.init(project="tpuchat-ablations",
           name=f"hero-{MLP_TYPE}-{ATTN_IMPL}-qknorm{QK_NORM}",
           config={
               "mlp_type": MLP_TYPE, "attn_impl": ATTN_IMPL, "qk_norm": QK_NORM,
               "learning_rate": hero_config.learning_rate,
               "non_embed_params": hero_non_embed,
               "target_tokens": target_tokens, "steps": HERO_STEPS,
           })
wandb.define_metric("train/loss", step_metric="step")
wandb.define_metric("train/tok_per_sec", step_metric="step")
wandb.define_metric("train/mfu_pct", step_metric="step")
wandb.define_metric("val/loss", step_metric="step")

smooth_loss = 0.0
best_val_loss = float('inf')
total_training_time = 0.0
params = hero_params

if SAVE_CHECKPOINTS:
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f'\n=== Hero Run: {HERO_STEPS:,} steps ===\n')

try:
    for step in range(HERO_STEPS + 1):
        last_step = (step == HERO_STEPS)

        # --- Eval ---
        if step % HERO_EVAL_EVERY == 0 or last_step:
            val_loader = val_loader_fn()
            val_losses = []
            for ei in range(hero_config.eval_steps):
                vx, vy = next(val_loader)
                vx, vy = jnp.array(vx), jnp.array(vy)
                vl = eval_step(hero_config, params, vx, vy)
                val_losses.append(float(vl))
            avg_val_loss = sum(val_losses) / len(val_losses)
            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss

            wandb.log({"step": step, "val/loss": avg_val_loss})
            print(f'Step {step:06d}/{HERO_STEPS} | Val loss: {avg_val_loss:.4f} '
                  f'(best: {best_val_loss:.4f})')

        # --- Checkpoint ---
        if SAVE_CHECKPOINTS and step > 0 and step % 50_000 == 0:
            import pickle as pkl
            ckpt_path = os.path.join(CHECKPOINT_DIR, f'params_step{step}.pkl')
            params_np = jax.tree.map(
                lambda x: np.array(x) if isinstance(x, jax.Array) else x, params)
            with open(ckpt_path, 'wb') as f:
                pkl.dump(params_np, f)
            print(f'  Checkpoint saved: {ckpt_path}')

        if last_step:
            break

        # --- Train step ---
        lr_mult = jnp.array(get_lr_multiplier(step, HERO_STEPS, hero_config),
                             dtype=jnp.float32)
        t0 = time.time()
        x_batch, y_batch = next(train_loader)
        loss, params, opt_state = train_step(hero_config, params, opt_state,
                                              x_batch, y_batch, lr_mult)
        loss.block_until_ready()
        dt = time.time() - t0

        if step > 20:
            total_training_time += dt

        loss_val = float(loss)
        ema_beta = 0.9
        smooth_loss = ema_beta * smooth_loss + (1 - ema_beta) * loss_val
        debiased_loss = smooth_loss / (1 - ema_beta ** (step + 1))

        if step % 1000 == 0:
            tok_per_sec = int(total_batch_size / dt) if dt > 0 else 0
            mfu_pct = step_flops / (PEAK_TFLOPS * 1e12 * dt) * 100 if dt > 0 else 0
            pct = 100 * step / HERO_STEPS
            eta = ''
            if step > 20 and total_training_time > 0:
                avg_dt = total_training_time / (step - 20)
                remaining = (HERO_STEPS - step) * avg_dt
                eta = f' | eta: {remaining/60:.0f}m'

            wandb.log({
                "step": step,
                "train/loss": debiased_loss,
                "train/tok_per_sec": tok_per_sec,
                "train/mfu_pct": mfu_pct,
            })
            print(f'step {step:06d}/{HERO_STEPS} ({pct:.1f}%) | '
                  f'loss: {debiased_loss:.4f} | MFU: {mfu_pct:.1f}% | '
                  f'tok/s: {tok_per_sec:,}{eta}')

finally:
    train_loader.stop()

wandb.finish()
print(f'\nHero run complete. Best val loss: {best_val_loss:.4f}')
print(f'Total training time: {total_training_time/3600:.1f}h')

# --- Sample text ---
print('\n--- Samples ---')
for prompt in ['The capital of France is', 'In a distant galaxy, scientists discovered',
               'Machine learning is']:
    text = generate(hero_config, params, prompt, max_new_tokens=100)
    print(f'Prompt: {prompt}\nOutput: {text}\n')